# N049 · 链表重排与回文

**目标：** 组合中点、反转和合并且明确是否恢复输入。

**先修：** N046, N047, N048。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 前后半段；交替拼接；回文比较；修改副作用。

## 从问题到算法

链表重排分成三步：找第一半末节点、反转第二半、交替穿插。判断回文只需反转后半并比较对应值；但检查不应该悄悄破坏调用者的数据，所以restore=True时还要反转回来并恢复原next关系。奇数长度的中间节点无需参与比较。

## 最小实现

**本章接口：** `reorder_list(head)`、`is_palindrome_list(head, restore=True)`

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def list_to_nodes(values):
    dummy = tail = ListNode()
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next

def nodes_to_list(head):
    values, seen = [], set()
    while head is not None:
        if id(head) in seen:
            raise ValueError('cycle in a supposedly finite list')
        seen.add(id(head))
        values.append(head.val)
        head = head.next
    return values
def reverse_list(head):
    previous=None
    while head:
        following=head.next
        head.next=previous
        previous,head=head,following
    return previous

def _first_half_end(head):
    slow=fast=head
    while fast.next and fast.next.next: slow=slow.next; fast=fast.next.next
    return slow

def reorder_list(head):
    if head is None or head.next is None: return
    middle=_first_half_end(head); second=reverse_list(middle.next); middle.next=None
    first=head
    while second:
        following1,following2=first.next,second.next
        first.next=second; second.next=following1
        first,second=following1,following2

def is_palindrome_list(head,restore=True):
    if head is None or head.next is None: return True
    middle=_first_half_end(head); second=reverse_list(middle.next); middle.next=second
    left,right=head,second; answer=True
    while right:
        if left.val!=right.val: answer=False
        left=left.next; right=right.next
    if restore: middle.next=reverse_list(second)
    return answer

## 正确性、前提与复杂度

后半反转后，其第i项对应原链倒数第i项。比较完整后再反转是原变换的逆操作，且保留中点前驱连接，因此恢复的是节点拓扑而非仅值。

**代价：** O(n)时间、O(1)辅助空间。restore=False明确允许后半次序改变。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

rows=[]
for a in [[1,2,3,4],[1,2,3,4,5],[1,2,2,1]]:
    h=list_to_nodes(a); palindrome=is_palindrome_list(h); reorder_list(h)
    rows.append((a,palindrome,nodes_to_list(h)))
show_table(['原链','是否回文','重排链'],rows)

原链,是否回文,重排链
"[1, 2, 3, 4]",False,"[1, 4, 2, 3]"
"[1, 2, 3, 4, 5]",False,"[1, 5, 2, 4, 3]"
"[1, 2, 2, 1]",True,"[1, 1, 2, 2]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert is_palindrome_list(list_to_nodes([1,2,2,1]))
assert not is_palindrome_list(list_to_nodes([1,2]))
from itertools import product
for n in range(7):
    for a in product([0,1],repeat=n):
        h=list_to_nodes(a); old={}; p=h
        while p: old[p]=p.next; p=p.next
        assert is_palindrome_list(h,restore=True)==(list(a)==list(a)[::-1])
        assert all(node.next is nxt for node,nxt in old.items())
        reorder_list(h)
        ref=[]; l=0; r=n-1
        while l<=r:
            ref.append(a[l]); l+=1
            if l<=r: ref.append(a[r]); r-=1
        assert nodes_to_list(h)==ref
print('N049: 所有本章断言通过')

N049: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 回文检查后恢复原链。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 用额外数组做独立参照。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 143. Reorder List（canonical）
- 234. Palindrome Linked List（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。